## LangChain创建检索增强生成应用
>RAG:通常是指检索增强生成（Retrieval-Augmented Generation），是一种将检索技术与语言生成模型相结合的方法，旨在提高语言生成的质量、准确性和相关性。


Load数据--->Split分割数据---> Embedding ---> STORE

In [1]:
import os

from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFaceEmbeddings

In [2]:
# 实例化模型
model_name = "qwen2.5:7b"
ollama_api_base = os.environ["OLLAMA_API_BASE"]

llm = init_chat_model(model=model_name, model_provider="ollama", base_url=ollama_api_base)

In [3]:
str_parser = StrOutputParser()

In [4]:
str_parser.invoke(llm.invoke("噪声抑制的作用?"))

'噪声抑制是一种信号处理技术，用于减少或消除语音或音频信号中的噪声干扰。其主要作用包括以下几个方面：\n\n1. **提高语音清晰度**：在电话、视频会议等场景中，噪声抑制可以显著提升通话质量，使得声音更加清晰可辨。\n\n2. **改善听觉体验**：对于耳机和音响设备来说，使用噪声抑制技术可以在一定程度上减少背景噪音对聆听体验的影响，提供更为纯净的音频效果。\n\n3. **增强语音识别准确率**：在智能音箱、语音助手等需要进行自动语音识别的应用中，有效的噪声抑制可以提高系统的识别准确性，从而提升用户体验。\n\n4. **医学应用**：在某些医学诊断设备中（如心电图），去除或减少干扰信号有助于更准确地获取关键数据。\n\n5. **音频处理与编辑**：在音乐制作、广播等领域，通过噪声抑制技术可以改进录音质量，消除不需要的背景噪音或其他杂音。\n\n实现噪声抑制的方法多种多样，包括但不限于：\n\n- **滤波器法**：如高通或低通滤波器可以帮助去除特定频率范围内的噪声。\n- **自适应滤波器**：这种高级方法能够根据环境变化自动调整参数以优化去噪效果。\n- **谱减法**：这种方法通过比较干净的声音和受污染的声音之间的频谱差异来减少噪声。\n\n使用适当的噪声抑制技术可以极大地改善各种音频应用中的声音质量，使得用户能够更好地享受高质量的语音或音乐体验。'

## 1. 准备数据

### 1.1. Loader
> pip install pdfminer.six

In [5]:
os.environ["USER_AGENT"] = "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)"

In [6]:
from langchain.vectorstores import FAISS
from langchain_community.document_loaders import PDFMinerLoader
from langchain.text_splitter import CharacterTextSplitter, RecursiveCharacterTextSplitter

In [7]:
# 1.1 加载PDF文件，注意文件的位置
loader = PDFMinerLoader("~/data/study/book/DHO800.pdf")
documents = loader.load()

### 1.2. Split

In [8]:
# 分割文本
# text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100, add_start_index=True)
texts = text_splitter.split_documents(documents=documents)

In [9]:
len(texts), type(texts[0])

(193, langchain_core.documents.base.Document)

### 1.3. Store
初始化向量模型，然后创建FAISS向量存储

In [10]:
# 3.1 初始化嵌入模型
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-large-en-v1.5")

In [11]:
# 3.2 创建FAISS向量存储
vectorstore = FAISS.from_documents(texts, embedding=embeddings)

In [12]:
# 进行相似性搜索
query = "噪声抑制的作用"
similar_docs = vectorstore.similarity_search(query)

# 输出相似文档内容
for doc in similar_docs:
    print("id:", doc.id, " | start_index:", doc.metadata['start_index'])

id: d42beea3-d99b-43dc-b27f-eb8d754085ea  | start_index: 95530
id: 16c5d64c-cf08-4a27-84dc-76f0daf12885  | start_index: 89263
id: 9aa6a213-82ac-4291-b88c-50497d0f53dd  | start_index: 104587
id: 6d3f264d-dc44-4449-a7ec-78e7fdf7232e  | start_index: 50489


In [13]:
# Retrieve
retriever = vectorstore.as_retriever()

## 2. 使用

In [14]:
from langchain import hub
# from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [15]:
prompt = hub.pull("rlm/rag-prompt")
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})])

In [16]:
# 查看prompt的示例
prompt.invoke({"context": "这个是文本内容", "question": "你的问题是?"}).to_messages()

[HumanMessage(content="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: 你的问题是? \nContext: 这个是文本内容 \nAnswer:", additional_kwargs={}, response_metadata={})]

In [17]:
def format_doc(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain =  (
    {"context": retriever | format_doc, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [18]:
# 查询问题
rag_chain.invoke("噪声抑制的作用?")

'噪声抑制的作用是减少或消除信号中的噪声干扰，使待测信号更清晰。'

再次对比一下，直接使用模型的回答。

In [19]:
StrOutputParser().invoke(llm.invoke("噪声抑制的作用?"))

'噪声抑制是信号处理中的一个重要技术，其主要作用包括以下几个方面：\n\n1. **改善通信质量**：在电信和无线通信领域，噪声抑制可以提高接收信号的质量。例如，在电话、无线电或卫星通信系统中，通过去除或减少背景噪音，能够使语音更加清晰。\n\n2. **音频增强**：在音响设备和家庭娱乐系统中，噪声抑制技术可以帮助提升音质，消除环境中的背景噪音对音频效果的影响，从而提供更舒适的听觉体验。\n\n3. **图像处理**：在图像信号处理领域，噪声抑制可以用于改善图像质量。通过去除或减少图像中的噪点（例如椒盐噪声、高斯噪声等），可以使图片看起来更加清晰和自然。\n\n4. **医疗成像**：在医学影像技术中，如MRI或CT扫描，噪声抑制可以帮助提高图像的对比度和细节，从而有助于医生更准确地诊断病情。\n\n5. **音频录音与处理**：对于音乐制作、广播以及电影音效等专业领域，噪声抑制同样发挥着重要作用。通过去除不必要的背景噪音，可以确保最终输出的声音作品质量更高。\n\n6. **语音识别与机器学习**：在语音信号处理中，噪声抑制能够提升语音识别系统的性能。它有助于减少环境噪音的影响，使机器更容易准确地识别和理解人类的语言输入。\n\n综上所述，噪声抑制技术广泛应用于各种需要处理或传输高质量音频、视频或其他形式的信号的场景之中。通过有效降低噪音干扰，这些技术不仅能提高通信质量和用户体验，还能在许多其他应用中发挥关键作用。'